# E111: one cold generation on free Colab GPU
Private operational benchmark under PLAN Phase 0. Use the coordinator's packet only after its CPU run and hosted tests finish. No training migration is authorized by this notebook.

The isolated `/content/seed-benchmark-venv` must contain the uploaded pinned requirements plus `jax[cuda13]==0.12.0.1`, with `pip check` passing. Keep package installation timing/output separately; installation is outside the generation timer. Never use the base Colab JAX environment. Python/OS/backend differences are recorded: this compares deployment environments, not GPU hardware in isolation.

Upload `Seed-E111.bundle` and `E111-manifest.json` from the coordinator's ready packet. Preserve failed artifacts; do not rerun the benchmark or change its settings after seeing timing. One generation includes cold JIT compilation. Compilation logs are diagnostic and must not be subtracted to invent warm throughput.


In [ ]:
from google.colab import files
from pathlib import Path
import datetime, hashlib, json, math, os, signal, subprocess, time, zipfile
files.upload()
packet = Path('/content/E111-manifest.json')
manifest = json.loads(packet.read_text())
bundle = Path('/content/Seed-E111.bundle')
assert hashlib.sha256(bundle.read_bytes()).hexdigest() == manifest['bundle_sha256']
assert manifest['cpu']['status'] == 'finished' and manifest['cpu']['generation'] == 1
assert manifest['cpu']['source_commit'] == manifest['source_commit']
assert manifest['ci']['conclusion'] == 'success'
assert len(manifest['source_commit']) == 40
repo = Path('/content/seed-e111')
assert not repo.exists(), 'Preserve the existing checkout and inspect before retrying.'
subprocess.run(['git', 'clone', str(bundle), str(repo)], check=True)
subprocess.run(['git', 'checkout', '--detach', manifest['source_commit']], cwd=repo, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip() == manifest['source_commit']
assert not subprocess.check_output(['git', 'status', '--porcelain'], cwd=repo, text=True).strip()
PYTHON = '/content/seed-benchmark-venv/bin/python'
assert Path(PYTHON).is_file()
evidence = repo / '.tmp/E111-gpu'
evidence.mkdir(parents=True, exist_ok=False)
env = dict(os.environ, JAX_PLATFORMS='cuda', JAX_ENABLE_COMPILATION_CACHE='false',
           JAX_LOG_COMPILES='1', XLA_PYTHON_CLIENT_PREALLOCATE='false', PYTHONUTF8='1')
env.pop('LD_LIBRARY_PATH', None)
print('Source:', manifest['source_commit'])


In [ ]:
import shutil
def available_memory():
    return int(next(line.split()[1] for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemAvailable:'))) * 1024
assert available_memory() >= 8 * 1024**3
assert shutil.disk_usage('/content').free >= 10 * 1024**3
preflight_started = time.perf_counter()
check = subprocess.run([PYTHON, '-m', 'pip', 'check'], capture_output=True, text=True)
(evidence / 'pip-check.txt').write_text(check.stdout + check.stderr)
check.check_returncode()
probe = """import json, platform
from pathlib import Path
from importlib.metadata import version
import jax
for line in Path('requirements.txt').read_text().splitlines():
    if line and not line.startswith('#'):
        name, expected = line.split('==')
        assert version(name) == expected, (name, version(name), expected)
assert version('jax-cuda13-plugin') == version('jax-cuda13-pjrt') == '0.12.0.1'
assert jax.default_backend() == 'gpu' and all(d.platform == 'gpu' for d in jax.devices())
assert not jax.config.jax_enable_compilation_cache
print(json.dumps(dict(python=platform.python_version(), platform=platform.platform(),
    backend=jax.default_backend(), devices=[str(d) for d in jax.devices()],
    jax=version('jax'), jaxlib=version('jaxlib'), numpy=version('numpy'))))
"""
probe_result = subprocess.run([PYTHON, '-c', probe], cwd=repo, env=env, capture_output=True, text=True, timeout=120)
(evidence / 'preflight-stderr.txt').write_text(probe_result.stderr)
probe_result.check_returncode()
preflight = json.loads(probe_result.stdout)
preflight['preflight_seconds'] = time.perf_counter() - preflight_started
preflight['available_memory_bytes'] = available_memory()
preflight['free_disk_bytes'] = shutil.disk_usage('/content').free
preflight['gpu_inventory'] = subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total,memory.free', '--format=csv,noheader'], text=True)
assert 'T4' in preflight['gpu_inventory'], preflight['gpu_inventory']
(evidence / 'preflight.json').write_text(json.dumps(preflight, indent=2))
(evidence / 'pip-freeze.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True))
for name in ('seed-setup.log', 'seed-setup-retry.log', 'seed-gpu-environment.json'):
    if Path('/content', name).exists():
        shutil.copyfile(Path('/content', name), evidence / name)
print(json.dumps(preflight, indent=2))


In [ ]:
assert not list((repo / 'runs').glob('*_E111-benchmark-gpu')), 'A prior attempt exists; preserve it.'
arguments = [PYTHON, '-X', 'utf8', 'scripts/run_experiment.py', '--name', 'E111-benchmark-gpu',
    '--start', 'file:runs/20261112-131946_E104-B-bio-s0/genome_start.npy',
    '--sigma', '0.05', '--lr', '0.02', '--generations', '1', '--seed', '2',
    '--pop', '128', '--worlds', '8', '--eval-worlds', '128', '--checkpoint-every', '10',
    '--max-train-seconds', '300']
started_at = datetime.datetime.now(datetime.timezone.utc).isoformat()
started = time.perf_counter()
timed_out = False
stop_reason = None
with (evidence / 'stdout.txt').open('x') as stdout, (evidence / 'compilation-stderr.txt').open('x') as stderr:
    process = subprocess.Popen(arguments, cwd=repo, env=env, stdout=stdout, stderr=stderr, start_new_session=True, preexec_fn=lambda: os.nice(10))
    try:
        while process.poll() is None:
            free_memory = available_memory()
            free_disk = shutil.disk_usage('/content').free
            elapsed = time.perf_counter() - started
            with (evidence / 'resources.jsonl').open('a') as samples:
                samples.write(json.dumps(dict(seconds=elapsed, available_memory_bytes=free_memory, free_disk_bytes=free_disk)) + '\n')
            if elapsed >= 600 or free_memory < 3 * 1024**3 or free_disk < 10 * 1024**3:
                stop_reason = 'Full-process time or runtime resource floor reached'
                raise subprocess.TimeoutExpired(arguments, elapsed)
            time.sleep(2)
        returncode = process.wait()
    except BaseException as error:
        timed_out = True
        stop_reason = stop_reason or f'{type(error).__name__}: {error}'
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGINT)
            except ProcessLookupError:
                pass
        try:
            returncode = process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            returncode = process.wait()
record = dict(source_commit=manifest['source_commit'], started_at=started_at,
              full_process_seconds=time.perf_counter()-started, returncode=returncode,
              interrupted_or_timed_out=timed_out, stop_reason=stop_reason, arguments=arguments)
(evidence / 'process.json').write_text(json.dumps(record, indent=2))
run_dirs = list((repo / 'runs').glob('*_E111-benchmark-gpu'))
archive = Path('/content/E111-gpu-results.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as zipped:
    for directory in [evidence, *run_dirs]:
        for path in directory.rglob('*'):
            if path.is_file():
                zipped.write(path, path.relative_to(repo))
print('Preserved results:', archive, json.dumps(record, indent=2))
assert returncode == 0 and not timed_out, 'Preserve and export the failed attempt; do not retry automatically.'
assert len(run_dirs) == 1
run = run_dirs[0]
summary = json.loads((run / 'summary.json').read_text())
config = json.loads((run / 'config.json').read_text())
rows = [json.loads(line) for line in (run / 'metrics.jsonl').read_text().splitlines()]
assert summary['status'] == 'finished' and summary['generation'] == 1
assert config['git']['commit'] == manifest['source_commit'] and not config['git']['dirty']
assert len(rows) == 1 and rows[0]['gen'] == 0 and math.isfinite(rows[0]['seconds']) and rows[0]['seconds'] > 0
ratio = manifest['cpu']['cold_generation_seconds'] / rows[0]['seconds']
print('Cold GPU seconds/generation (includes compilation):', rows[0]['seconds'])
print('Cold CPU/GPU ratio:', ratio)
print('At least 5x: propose migration and wait for owner.' if ratio >= 5 else 'Below 5x: retain laptop training under the registered decision rule.')
print('No uncertainty estimate or warm-throughput claim is supported by one measurement per platform.')


In [ ]:
files.download('/content/E111-gpu-results.zip')
